# 🗣️ Orpheus-3B — Zero-Shot Voice Cloning

Clone a voice from a **20–30 second** sample and make it speak a hard-coded
sentence, using **`unsloth/orpheus-3b-0.1-pretrained`** (a Llama-3.2-3B based
Speech-LLM) + the **SNAC** 24 kHz neural audio codec.

**How it works (the honest version).** Orpheus is a *completion* model over a
joint text+audio token vocabulary. There is no separate "speaker embedding"
API. Zero-shot cloning works by **in-context conditioning**: we feed the model a
completed turn — *(reference transcript → reference audio tokens)* — and then a
new turn with the target text, and let it *continue* in the same voice. This is
the mechanism Orpheus is documented to support, and the **pretrained** base
checkpoint (not the `-ft` one) is the right model for it, because in-context
continuation is exactly what a base model does. The `-ft` checkpoint is tuned
for a fixed set of named speakers and is a worse fit for cloning — it is offered
only as a *fallback for normal TTS* at the bottom of this notebook.

> ⚠️ **Limitations — read these.** Zero-shot quality from the base model is
> **variable**. You will get good results only with (1) a **clean** reference
> (no music/noise, single speaker), (2) an **accurate** `REF_TRANSCRIPT` that
> matches the words actually spoken, and (3) a 24 kHz mono sample. Expect to try
> a few reference clips / random seeds. This is a real limitation of the
> approach, not a bug in the notebook.

---
## ✋ Consent & responsible use
Only clone voices you **own** or have **explicit permission** to clone. Do not
use this to impersonate, deceive, or create misleading content. Cloning a
person's voice without consent may be illegal in your jurisdiction.

---
## ▶️ Running this in Google Colab
1. **Runtime → Change runtime type → GPU** (T4 is enough with 4-bit).
2. Get the notebook into Colab — any one of:
   - **File → Upload notebook** and pick this `.ipynb`, **or**
   - **File → Open notebook → GitHub** tab, paste your repo URL, **or**
   - clone the repo from a cell: `!git clone https://github.com/<you>/<repo>.git`
     then open the `.ipynb` from the file browser on the left.
3. Run the cells **top to bottom**. The first install cell takes a few minutes.


## 1. Setup and installation
Installs Unsloth (efficient model loading + 4-bit), the SNAC codec, and audio
I/O libraries. Safe to re-run. On Colab `ffmpeg` is preinstalled (needed for
MP3/M4A decoding).

In [ ]:
# --- Install dependencies (Colab or fresh local env) ---------------------
# %%capture  # uncomment to hide pip noise
import sys, subprocess, shutil

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Unsloth pulls compatible torch/transformers/bitsandbytes for the environment.
pip_install("unsloth")
# Audio codec + I/O.
pip_install("snac", "soundfile", "librosa")

# ffmpeg is required to decode MP3/M4A. It ships with Colab; warn if missing.
if shutil.which("ffmpeg") is None:
    print("⚠️ ffmpeg not found. WAV will still work, but MP3/M4A may not.")
    print("   Colab: run  !apt-get -y install ffmpeg")
    print("   macOS: brew install ffmpeg   |   Ubuntu: sudo apt install ffmpeg")
else:
    print("✅ ffmpeg found:", shutil.which("ffmpeg"))
print("✅ Install step finished.")

## 2. Imports
Also verifies a CUDA GPU is available — Unsloth + Orpheus require one.

In [ ]:
import os, gc, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")

# --- GPU check ----------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. This notebook needs a GPU.\n"
        "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("   torch:", torch.__version__)

## 3. Configuration
Everything you might change lives here.

**You MUST set `REF_TRANSCRIPT`** to the exact words spoken in your reference
clip — accurate text is the single biggest factor in clone quality.

In [ ]:
# --- Model -------------------------------------------------------------------
MODEL_NAME      = "unsloth/orpheus-3b-0.1-pretrained"  # base model for cloning
LOAD_IN_4BIT    = True    # 4-bit (bnb) -> ~fits a T4. Set False on big GPUs for
                          # slightly higher quality at higher VRAM cost.
MAX_SEQ_LENGTH  = 8192    # MUST hold ref-audio tokens (~150/sec -> ~4500 for 30s)
                          # + target generation. Do NOT lower below ~6000.

# --- Reference voice ---------------------------------------------------------
REF_AUDIO_PATH  = ""      # leave "" to upload in Colab, or set a local path here
# ⬇️ EXACTLY what is said in the reference clip (clean punctuation helps):
REF_TRANSCRIPT  = "Replace this text with the exact words spoken in your reference audio sample."

# --- What the cloned voice should say (hard-coded, per requirements) ----------
TARGET_TEXT = ("Hello, this is a cloned voice test using Orpheus. The model "
               "should speak this sentence in the style of the uploaded voice sample.")

# --- Audio preprocessing -----------------------------------------------------
TARGET_SAMPLE_RATE = 24000   # Orpheus / SNAC operate at 24 kHz
MIN_DURATION_SEC   = 20      # target window per requirements
MAX_DURATION_SEC   = 30
HARD_MIN_SEC       = 15      # below this we warn loudly
TRIM_TOP_DB        = 30      # silence-trim aggressiveness (higher = less trimming)
PEAK_NORM          = 0.95    # safe peak normalization target

# --- Generation --------------------------------------------------------------
TEMPERATURE        = 0.6
TOP_P              = 0.95
REPETITION_PENALTY = 1.1     # >= 1.1 recommended by Orpheus for stable audio
MAX_NEW_TOKENS     = 1200    # covers the TARGET sentence only (~8s audio). Raise
                             # if the target text gets cut off mid-sentence.
PRIME_SPEECH       = True    # End the cloning prompt with [SOAI][SOS] to force the
                             # model straight into target audio. This is the one
                             # part NOT copied from verified code (the two-turn
                             # cloning prompt is our own construction). If output
                             # is EMPTY / garbled / not in the cloned voice, try
                             # flipping this to False (let the model emit SOAI/SOS).

OUTPUT_PATH = "output_cloned_voice.wav"
print("Config loaded. Target text:\n ", TARGET_TEXT)

## 4. Upload / load voice sample
In Colab a file picker appears. Locally (or to skip the picker), set
`REF_AUDIO_PATH` in the config cell. Accepts WAV/MP3/M4A/FLAC.

In [ ]:
# Detect Colab and offer an upload widget if no path was configured.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not REF_AUDIO_PATH:
    if IN_COLAB:
        print("Upload your 20-30s voice sample (WAV/MP3/M4A/FLAC):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        REF_AUDIO_PATH = list(uploaded.keys())[0]
    else:
        raise RuntimeError(
            "REF_AUDIO_PATH is empty and not running in Colab.\n"
            "Set REF_AUDIO_PATH in the Configuration cell to your audio file."
        )

if not os.path.exists(REF_AUDIO_PATH):
    raise FileNotFoundError(f"Audio file not found: {REF_AUDIO_PATH}")
print("✅ Reference audio:", REF_AUDIO_PATH)

## 5. Audio preprocessing
Loads the file, converts to **mono @ 24 kHz**, trims leading/trailing silence,
peak-normalizes safely, and validates the duration is in the 20–30 s window.

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

def load_and_preprocess(path):
    """Return a float32 mono waveform at TARGET_SAMPLE_RATE, cleaned up."""
    # librosa.load handles WAV/MP3/M4A/FLAC (via soundfile / audioread+ffmpeg),
    # downmixes to mono, and resamples to the requested rate.
    try:
        wav, sr = librosa.load(path, sr=TARGET_SAMPLE_RATE, mono=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not decode '{path}'. Unsupported format or missing ffmpeg.\n"
            f"Original error: {e}"
        )

    # Trim silence from both ends.
    wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)

    # Safe peak normalization (avoid divide-by-zero on silent clips).
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 0:
        wav = (wav / peak) * PEAK_NORM
    wav = wav.astype(np.float32)
    return wav, TARGET_SAMPLE_RATE

ref_wav, ref_sr = load_and_preprocess(REF_AUDIO_PATH)
duration = len(ref_wav) / ref_sr
print(f"Processed: {duration:.1f}s, mono, {ref_sr} Hz, {len(ref_wav)} samples")

# Duration validation / warnings.
if duration < HARD_MIN_SEC:
    print(f"🚨 WARNING: only {duration:.1f}s — too short. Cloning will be poor. "
          f"Aim for {MIN_DURATION_SEC}-{MAX_DURATION_SEC}s.")
elif duration < MIN_DURATION_SEC:
    print(f"⚠️ {duration:.1f}s is a little short; {MIN_DURATION_SEC}-{MAX_DURATION_SEC}s is ideal.")
elif duration > MAX_DURATION_SEC:
    print(f"⚠️ {duration:.1f}s is longer than {MAX_DURATION_SEC}s; will use it but it "
          f"costs context length. Consider trimming.")
else:
    print("✅ Duration is in the ideal 20-30s window.")

print("Preview of your processed reference:")
display(Audio(ref_wav, rate=ref_sr))

## 6. Model loading
Loads Orpheus via Unsloth (optionally 4-bit) and the SNAC codec. Includes an
out-of-memory fallback hint.

In [ ]:
# Robust loader: try FastLanguageModel first (its from_pretrained explicitly
# accepts max_seq_length/dtype/load_in_4bit), fall back to FastModel if the
# installed Unsloth version routes Orpheus differently.
from unsloth import FastLanguageModel
try:
    from unsloth import FastModel
except Exception:
    FastModel = None

model = tokenizer = None
load_err = None
for loader_name, loader in [("FastLanguageModel", FastLanguageModel),
                            ("FastModel", FastModel)]:
    if loader is None:
        continue
    try:
        model, tokenizer = loader.from_pretrained(
            model_name      = MODEL_NAME,
            max_seq_length  = MAX_SEQ_LENGTH,
            dtype           = None,          # auto: fp16 on T4/V100, bf16 on Ampere+
            load_in_4bit    = LOAD_IN_4BIT,  # bnb 4-bit quantization
        )
        print(f"✅ Loaded via {loader_name}.")
        break
    except torch.cuda.OutOfMemoryError:
        raise RuntimeError(
            "CUDA OOM while loading the model.\n"
            "Fixes: set LOAD_IN_4BIT = True, restart the runtime, or use a bigger GPU."
        )
    except TypeError as e:
        # Signature mismatch for this loader — try the next one.
        load_err = e
        continue
    except Exception as e:
        raise RuntimeError(
            f"Model download/load failed: {e}\n"
            "If the model is gated, run `huggingface-cli login` (or set HF_TOKEN) and "
            "accept the model terms on its Hugging Face page."
        )

if model is None:
    raise RuntimeError(f"Could not load {MODEL_NAME} with any Unsloth loader. "
                       f"Last error: {load_err}")

# Enable Unsloth's fast inference path (best-effort across versions).
try:
    FastLanguageModel.for_inference(model)
except Exception:
    pass
model.eval()
print("✅ Orpheus ready.")

# --- SNAC audio codec (encodes the reference, decodes the output) -------------
from snac import SNAC
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda").eval()
print("✅ SNAC 24kHz loaded.")

## 7. Voice cloning / speaker conditioning
We encode the reference waveform into SNAC tokens using Orpheus's exact 7-tokens-
per-frame layout (constants taken verbatim from the official Unsloth Orpheus
notebook). These tokens become the in-context "voice prompt".

In [ ]:
import torchaudio.transforms as T

# --- Orpheus special-token constants (verbatim from the official notebook) ----
tokeniser_length = 128256
start_of_text  = 128000
end_of_text    = 128009
start_of_speech = tokeniser_length + 1   # 128257
end_of_speech   = tokeniser_length + 2   # 128258
start_of_human  = tokeniser_length + 3   # 128259
end_of_human    = tokeniser_length + 4   # 128260
start_of_ai     = tokeniser_length + 5   # 128261
end_of_ai       = tokeniser_length + 6   # 128262
pad_token       = tokeniser_length + 7   # 128263
audio_tokens_start = tokeniser_length + 10  # 128266

def tokenise_audio(waveform_np, orig_sr=TARGET_SAMPLE_RATE):
    """Encode a mono float32 waveform into Orpheus audio tokens (7 per frame)."""
    wav = torch.from_numpy(waveform_np).unsqueeze(0).to(torch.float32)
    if orig_sr != 24000:  # SNAC expects 24 kHz
        wav = T.Resample(orig_freq=orig_sr, new_freq=24000)(wav)
    wav = wav.unsqueeze(0).to("cuda")  # shape [1, 1, T]
    with torch.inference_mode():
        codes = snac_model.encode(wav)
    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item()       + 128266)
        all_codes.append(codes[1][0][2*i].item()     + 128266 + 4096)
        all_codes.append(codes[2][0][4*i].item()     + 128266 + (2*4096))
        all_codes.append(codes[2][0][(4*i)+1].item() + 128266 + (3*4096))
        all_codes.append(codes[1][0][(2*i)+1].item() + 128266 + (4*4096))
        all_codes.append(codes[2][0][(4*i)+2].item() + 128266 + (5*4096))
        all_codes.append(codes[2][0][(4*i)+3].item() + 128266 + (6*4096))
    return all_codes

ref_codes = tokenise_audio(ref_wav, orig_sr=TARGET_SAMPLE_RATE)
# ~7 tokens/frame, ~150 tokens/sec of audio at 24 kHz.
print(f"Reference encoded: {len(ref_codes)} tokens ({len(ref_codes)//7} frames, "
      f"~{len(ref_codes)/150:.1f}s of codec audio)")
if len(ref_codes) % 7 != 0:
    print("⚠️ token count not divisible by 7 — unexpected; check the reference audio.")

## 8. Text-to-speech generation
We build the in-context cloning prompt:

```
[SOH] ref_transcript [EOT][EOH] [SOAI][SOS] ref_audio_tokens [EOS][EOAI]   <- the voice example
[SOH] target_text    [EOT][EOH] [SOAI][SOS] ...generate from here...       <- continue in that voice
```

Then we parse the generated stream exactly as the official notebook does: keep
everything after the **last** `start_of_speech`, drop `end_of_speech`, trim to a
multiple of 7, subtract the `128266` audio offset, and SNAC-decode.

In [ ]:
# --- Build the cloning prompt -------------------------------------------------
ref_text_ids = tokenizer.encode(REF_TRANSCRIPT, add_special_tokens=True)  # adds BOS
ref_text_ids.append(end_of_text)
tgt_text_ids = tokenizer.encode(TARGET_TEXT, add_special_tokens=True)
tgt_text_ids.append(end_of_text)

# Turn 1 = the voice example (transcript + its audio). Turn 2 = target text.
prompt_ids = (
    [start_of_human] + ref_text_ids + [end_of_human]
    + [start_of_ai] + [start_of_speech] + ref_codes + [end_of_speech] + [end_of_ai]
    + [start_of_human] + tgt_text_ids + [end_of_human]
)
# Optionally prime the AI turn so generation goes straight into target audio
# (see PRIME_SPEECH in the config cell — flip it if results are empty/garbled).
if PRIME_SPEECH:
    prompt_ids += [start_of_ai, start_of_speech]

input_ids = torch.tensor([prompt_ids], dtype=torch.int64).to("cuda")
attention_mask = torch.ones_like(input_ids)
print(f"Prompt length: {input_ids.shape[1]} tokens (max_seq_length={MAX_SEQ_LENGTH})")
if input_ids.shape[1] + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
    print("⚠️ prompt + generation may exceed max_seq_length. Use a shorter reference "
          "or raise MAX_SEQ_LENGTH and reload the model.")

# --- Generate ----------------------------------------------------------------
try:
    with torch.inference_mode():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            num_return_sequences=1,
            eos_token_id=end_of_speech,
            use_cache=True,
        )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError(
        "CUDA OOM during generation.\n"
        "Fixes: use a shorter reference clip (closer to 20s), lower MAX_NEW_TOKENS, "
        "keep LOAD_IN_4BIT=True, or restart the runtime."
    )

# --- Parse generated tokens (verbatim logic from the official notebook) -------
token_to_find, token_to_remove = 128257, 128258  # start_of_speech, end_of_speech
idx = (generated_ids == token_to_find).nonzero(as_tuple=True)
if len(idx[1]) > 0:
    last = idx[1][-1].item()
    cropped = generated_ids[:, last + 1:]   # keep only the TARGET audio
else:
    cropped = generated_ids

row = cropped[0]
row = row[row != token_to_remove]           # drop end_of_speech
new_len = (row.size(0) // 7) * 7            # trim to whole frames
row = row[:new_len]
code_list = [t.item() - 128266 for t in row]  # remove audio offset

if len(code_list) == 0:
    raise RuntimeError(
        "No audio tokens were generated. Try re-running (sampling varies), toggling "
        "PRIME_SPEECH, increasing MAX_NEW_TOKENS, or improving REF_TRANSCRIPT accuracy."
    )
print(f"✅ Generated {len(code_list)} audio tokens ({len(code_list)//7} frames).")

## 9. Save and play output audio
Decode SNAC tokens to a waveform, save `output_cloned_voice.wav`, and show a
player.

In [ ]:
def redistribute_codes(code_list):
    """Inverse of tokenise_audio: split flat tokens into SNAC's 3 layers, decode."""
    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(code_list) + 1) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - (2*4096))
        layer_3.append(code_list[7*i + 3] - (3*4096))
        layer_2.append(code_list[7*i + 4] - (4*4096))
        layer_3.append(code_list[7*i + 5] - (5*4096))
        layer_3.append(code_list[7*i + 6] - (6*4096))
    codes = [torch.tensor(layer_1).unsqueeze(0),
             torch.tensor(layer_2).unsqueeze(0),
             torch.tensor(layer_3).unsqueeze(0)]
    with torch.inference_mode():
        audio_hat = snac_model.decode(codes)  # decode on CPU below to save VRAM
    return audio_hat

# Move SNAC to CPU for decoding (frees VRAM, avoids OOM right after generation).
snac_model.to("cpu")
audio_hat = redistribute_codes(code_list)
audio_np = audio_hat.detach().squeeze().cpu().numpy().astype(np.float32)

import soundfile as sf
sf.write(OUTPUT_PATH, audio_np, TARGET_SAMPLE_RATE)
print(f"✅ Saved {OUTPUT_PATH}  ({len(audio_np)/TARGET_SAMPLE_RATE:.1f}s)")

from IPython.display import Audio, display
print("🔊 Cloned voice saying the target text:")
display(Audio(audio_np, rate=TARGET_SAMPLE_RATE))

# In Colab, uncomment to download:
# from google.colab import files; files.download(OUTPUT_PATH)

## 10. Optional fallback — named-voice TTS (NOT cloning)

If cloning quality is unacceptable or the pretrained model fails for you, this
runs **standard** Orpheus TTS with the fine-tuned checkpoint
`unsloth/orpheus-3b-0.1-ft` and a built-in named voice (e.g. `tara`).

> ⚠️ **This is NOT your cloned voice.** It speaks `TARGET_TEXT` in a generic,
> pre-trained speaker. It is only a sanity-check / fallback so you still get
> usable TTS. The `-ft` checkpoint is tuned for fixed speakers and is a poor fit
> for in-context cloning — that is why cloning above uses the *pretrained* model.

Available voices: `tara, leah, jess, leo, dan, mia, zac, zoe`.

In [ ]:
# ====== OPTIONAL FALLBACK — standard TTS, named voice (run only if needed) =====
RUN_FALLBACK = False   # set True to execute this cell's body

if RUN_FALLBACK:
    chosen_voice = "tara"
    FT_MODEL = "unsloth/orpheus-3b-0.1-ft"

    # Free the cloning model first — two 3B models won't fit a T4 together.
    try:
        del model, tokenizer
    except NameError:
        pass
    gc.collect(); torch.cuda.empty_cache()

    # Reuse SNAC (move back to GPU for encode/decode); load the ft model.
    snac_model.to("cuda")
    from unsloth import FastLanguageModel
    ft_model, ft_tok = FastLanguageModel.from_pretrained(
        model_name=FT_MODEL, max_seq_length=4096, dtype=None, load_in_4bit=LOAD_IN_4BIT)
    try:
        FastLanguageModel.for_inference(ft_model)
    except Exception:
        pass
    ft_model.eval()

    # ft prompt format is simply "{voice}: {text}".
    prompt = f"{chosen_voice}: {TARGET_TEXT}"
    ids = ft_tok(prompt, return_tensors="pt").input_ids
    soh = torch.tensor([[128259]], dtype=torch.int64)
    eoh = torch.tensor([[128009, 128260]], dtype=torch.int64)  # EOT, EOH
    ids = torch.cat([soh, ids, eoh], dim=1).to("cuda")
    am = torch.ones_like(ids)

    with torch.inference_mode():
        gen = ft_model.generate(input_ids=ids, attention_mask=am, max_new_tokens=1200,
                                do_sample=True, temperature=0.6, top_p=0.95,
                                repetition_penalty=1.1, eos_token_id=128258, use_cache=True)

    # Same parse + decode as the main path.
    i2 = (gen == 128257).nonzero(as_tuple=True)
    crop = gen[:, i2[1][-1].item() + 1:] if len(i2[1]) > 0 else gen
    r = crop[0]; r = r[r != 128258]; r = r[:(r.size(0)//7)*7]
    cl = [t.item() - 128266 for t in r]
    snac_model.to("cpu")
    out = redistribute_codes(cl).detach().squeeze().cpu().numpy().astype(np.float32)
    sf.write("output_fallback_tts.wav", out, TARGET_SAMPLE_RATE)
    print(f"✅ Fallback saved output_fallback_tts.wav (voice='{chosen_voice}')")
    display(Audio(out, rate=TARGET_SAMPLE_RATE))
else:
    print("Fallback skipped. Set RUN_FALLBACK = True to run named-voice TTS.")

## 11. Troubleshooting notes

| Symptom | Fix |
|---|---|
| **No GPU detected** | Colab: *Runtime → Change runtime type → GPU*. Locally you need an NVIDIA GPU + CUDA. |
| **`ffmpeg` errors / can't read MP3** | Install ffmpeg (`!apt-get -y install ffmpeg` on Colab) or convert the sample to WAV first. |
| **Model download fails / gated** | `huggingface-cli login` (or set `HF_TOKEN`) and accept the model terms on its HF page. |
| **CUDA out of memory (loading)** | Keep `LOAD_IN_4BIT = True`, restart runtime, or use a larger GPU. |
| **CUDA OOM (generation)** | Use a shorter reference (~20s), lower `MAX_NEW_TOKENS`, restart runtime. |
| **Garbled / wrong-content audio** | Make `REF_TRANSCRIPT` *exactly* match the spoken words; use a clean single-speaker clip; re-run (sampling varies); try a different reference. |
| **Voice doesn't resemble the sample** | Inherent to zero-shot from the base model. Use a longer/cleaner reference, try several seeds, or expressive reference speech. Consider LoRA fine-tuning (below) for production. |
| **Prompt exceeds `max_seq_length`** | Reduce reference length or raise `MAX_SEQ_LENGTH` and reload the model. |
| **Repetitive / looping audio** | Keep `REPETITION_PENALTY >= 1.1`; lower `TEMPERATURE` slightly. |

### Going further: LoRA fine-tuning for a stronger clone
For consistent, production-grade cloning of *one* voice, in-context zero-shot is
not the strongest option — a short **LoRA fine-tune** on that speaker is. Unsloth
provides the canonical recipe; format each example as
`[SOH] text [EOT][EOH] [SOAI][SOS] audio_tokens [EOS][EOAI]` (the
`create_input_ids` format used above) over several minutes of that speaker, then
infer with the prompt format from the fallback cell. See the Unsloth TTS
fine-tuning docs and the official `Orpheus_(3B)-TTS.ipynb`.
